# Fine-tune Llama 3.2 1B Instruct for congenital toxoplasmosis classification

Local, on-GPU fine-tuning notebook. Governed by `specs/001-toxo-graphrag-assistant/`:
`research.md` R1 and R3, `contracts/corpus-split.md`, and FR-084, FR-087, FR-097, FR-101,
FR-105–FR-107.

**What this notebook does and does not do**

- Trains a LoRA adapter on `meta-llama/Llama-3.2-1B-Instruct` to emit **only the classification
  labels** (`mother_classification`, `child_classification`) for a set of structured findings.
  The canonical Portuguese argumentation and recommendation are *never* generated by this model —
  they are rendered at serving time from a checked-in lookup keyed by the classification pair
  (research.md R1). This notebook does not touch that lookup.
- Loads `eval/split.v1.json` and calls `eval/dataset.py` directly. It **never regenerates the
  split** — that artefact is the single source of truth for what is training material and what is
  held out, and it is committed separately.
- Trains on the 13 training tuples (19 rows) only. Loading any of the 5 held-out test tuples is a
  hard error, not a filtered warning, checked explicitly below in addition to the check already
  inside `eval/dataset.py`.
- Runs **only** on this machine's GPU — an NVIDIA GeForce RTX 2070 Mobile with 8 GB of VRAM
  (FR-105). No cloud training service is used, and the cell below refuses to continue on CPU
  rather than silently training somewhere the spec doesn't sanction.
- Saves the merged model under `training_model/llama32-1b-toxo/` (git-ignored — FR-106) and
  pushes it to a **private** Hugging Face repository (FR-097). Only the finished weights leave
  this machine.

**What this notebook explicitly does not do**: it does not run the SC-015 replay, the SC-023
held-out evaluation, or the SC-016 determinism check. Those are `eval/replay.py`,
`eval/heldout.py`, and `eval/determinism.py` — reproducible command-line scripts, run against the
Hugging Face Inference Endpoint once it is configured (`eval/ENDPOINT.md`), not manually executed
notebook cells (plan.md Structure Decision).

**Before running**: `pip install -r training_model/requirements.txt` (after installing the CUDA
build of `torch` that matches your driver — see that file), and make sure `huggingface-cli login`
or a cached token can read `meta-llama/Llama-3.2-1B-Instruct` (accept the license on its model
page first; it is gated) and can create a private repository.

Run top to bottom. Every hyperparameter lives in the Config cell below.


## 1. Environment check — this must run on the local GPU, not CPU or a cloud notebook

In [ ]:
import platform
import sys

import torch

print(f"Python:       {sys.version.split()[0]}")
print(f"Platform:     {platform.platform()}")
print(f"torch:        {torch.__version__}")
print(f"torch CUDA:   {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available to torch, so this notebook is refusing to continue.\n"
        "FR-105 requires fine-tuning to run on the local GPU (RTX 2070 Mobile, 8 GB) — training\n"
        "on CPU here silently violates that constraint and would be extremely slow besides, and\n"
        "FR-107 forbids moving training to a cloud service instead.\n\n"
        "Common cause on a fresh boot / driver upgrade: the nvidia kernel module is installed\n"
        "(check `dkms status`) but not loaded (`lsmod | grep nvidia` shows nothing). With Secure\n"
        "Boot enabled this usually means the module needs to be loaded via a reboot (and, if it\n"
        "still doesn't load, MOK-enrolling the DKMS signing key). Reboot, confirm `nvidia-smi`\n"
        "reports the GPU from a terminal, then re-run this notebook from the top."
    )

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"\nGPU:          {device_name}")
print(f"VRAM:         {vram_gb:.1f} GB")

if vram_gb < 7.0:
    print(
        "\nWARNING: less than 7 GB of VRAM visible. research.md R3 sizes this run for an 8 GB\n"
        "card; if training runs out of memory, lower PER_DEVICE_BATCH_SIZE or MAX_SEQ_LEN in the\n"
        "Config cell below before increasing gradient accumulation."
    )


## 2. Config — every tunable in one place

In [ ]:
from pathlib import Path

# --- Paths -------------------------------------------------------------------------------------
REPO_ROOT = Path.cwd().resolve().parent  # training_model/ is a direct child of the repo root
EVAL_DIR = REPO_ROOT / "eval"
SPLIT_PATH = EVAL_DIR / "split.v1.json"
SOURCE_CSV = REPO_ROOT / "logs" / "request-logs.csv"
OUTPUT_DIR = Path.cwd() / "llama32-1b-toxo"          # git-ignored (FR-106)
CHECKPOINT_DIR = Path.cwd() / "checkpoints"          # git-ignored, intermediate only

# --- Model ---------------------------------------------------------------------------------------
BASE_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"    # FR-084's floor of the 1B -> 3B -> 8B ladder
MAX_SEQ_LEN = 512                                     # generous for a 14-field findings prompt

# --- Hugging Face Hub ------------------------------------------------------------------------
# None = auto-detect the logged-in user via `huggingface_hub.whoami()` and publish to
# f"{username}/toxoai-llama32-1b-ft". Set explicitly to override.
HF_REPO_ID = None
HF_REPO_PRIVATE = True                                # FR-097 — never a public repository

# --- LoRA (research.md R3: r=8-16, low rank, small learning rate) ----------------------------
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
]

# --- Training ------------------------------------------------------------------------------------
SEED = 42
LEARNING_RATE = 2e-4
NUM_EPOCHS = 15
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 2                                  # effective batch size 4
WARMUP_RATIO = 0.03

# 13 training tuples (19 rows) is too small to comfortably carve out a validation slice without
# starving training further, so the default is a fixed epoch count (research.md R3's documented
# fallback). Flip this on to instead hold back a few TRAINING rows (never test rows) for early
# stopping.
USE_VALIDATION_SLICE = False
VALIDATION_ROWS = 3   # only used if USE_VALIDATION_SLICE is True

print(f"Base model:      {BASE_MODEL_ID}")
print(f"Output dir:      {OUTPUT_DIR}")
print(f"LoRA:            r={LORA_R} alpha={LORA_ALPHA} dropout={LORA_DROPOUT}")
print(f"Epochs:          {NUM_EPOCHS} (fixed count)" if not USE_VALIDATION_SLICE
      else f"Epochs:          up to {NUM_EPOCHS}, early stopping on a {VALIDATION_ROWS}-row validation slice")
print(f"Effective batch: {PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS}")


## 3. Load the fixed split and the training dataset

`eval/dataset.py` is imported directly rather than reading an intermediate file — the split and
the source CSV stay the single source of truth. This cell also re-asserts, independently of the
check already inside `dataset.py`, that no held-out tuple has leaked in (FR-101).


In [ ]:
import sys

sys.path.insert(0, str(EVAL_DIR))
import dataset as dataset_mod  # eval/dataset.py

import json

split = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
print(f"Split version:      {split['version']}")
print(f"Source sha256:      {split['source_sha256'][:16]}...")
print(f"Train tuples/rows:  {split['totals']['train_tuples']} / {split['totals']['train_rows']}")
print(f"Test tuples/rows:   {split['totals']['test_tuples']} / {split['totals']['test_rows']}")
print(f"unmeasured_classes: {split['unmeasured_classes']}")
print(
    "\nNOTE (Principle VII): the six classes above have exactly one distinct input tuple each.\n"
    "They sit entirely in training and SC-023's held-out score will say nothing about them —\n"
    "that limitation is carried into every evaluation report downstream, not hidden here."
)

examples = dataset_mod.build_training_examples(SPLIT_PATH, SOURCE_CSV)
print(f"\nLoaded {len(examples)} training examples.")

test_record_ids = {rid for e in split["test"] for rid in e["record_ids"]}
train_record_ids = {e["record_id"] for e in examples}
leaked = test_record_ids & train_record_ids
assert not leaked, (
    f"Held-out record(s) {leaked} appear in the training examples — refusing to train. "
    "This must never happen (FR-101); eval/dataset.py already guards this internally, so seeing "
    "it here means the split artefact or the dataset writer has been changed inconsistently."
)
print("Confirmed: zero overlap between training examples and the held-out test set.")


In [ ]:
# Sanity-check one example end to end before it goes anywhere near the tokenizer.
example = examples[0]
print(f"record_id:  {example['record_id']}  (final_situation={example['final_situation']})")
print(f"prompt_version: {example['prompt_version']}")
print("\n--- prompt ---")
print(example["prompt"])
print("\n--- completion (training target — labels only, no argumentation/recommendation) ---")
print(example["completion"])


## 4. Build chat-formatted training text

Each example becomes a three-turn chat (system / user / assistant) using the base model's own
chat template, so the fine-tuned model matches the prompt shape it will actually be served with
(`backend_files/services/classifier.py`, T060 — that prompt builder must mirror
`eval/dataset.py`'s `PROMPT_VERSION`; a served prompt that disagrees with the trained one silently
degrades the model).


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # right-padding for training; left-padding is a serving concern

RESPONSE_TEMPLATE = "<|start_header_id|>assistant<|end_header_id|>\n\n"


def to_chat_text(example: dict) -> str:
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["completion"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)


chat_examples = [{**e, "text": to_chat_text(e)} for e in examples]
print(chat_examples[0]["text"])


In [ ]:
from datasets import Dataset

set_seed_value = SEED
import random

import numpy as np
from transformers import set_seed

random.seed(set_seed_value)
np.random.seed(set_seed_value)
set_seed(set_seed_value)

if USE_VALIDATION_SLICE:
    ordered = sorted(chat_examples, key=lambda e: int(e["record_id"]))
    if len(ordered) <= VALIDATION_ROWS:
        raise ValueError(
            f"Only {len(ordered)} training rows available — cannot carve out "
            f"{VALIDATION_ROWS} for validation and still have anything to train on. "
            "Set USE_VALIDATION_SLICE = False."
        )
    val_rows, train_rows = ordered[:VALIDATION_ROWS], ordered[VALIDATION_ROWS:]
    train_ds = Dataset.from_list(train_rows)
    eval_ds = Dataset.from_list(val_rows)
    print(f"Train rows: {len(train_rows)}   Validation rows (carved from TRAIN, never test): {len(val_rows)}")
else:
    train_ds = Dataset.from_list(chat_examples)
    eval_ds = None
    print(f"Train rows: {len(chat_examples)}   Validation: none (fixed epoch count)")


## 5. Load the base model and attach a LoRA adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM

# Turing (the RTX 2070 Mobile's architecture) has no bf16 support, so fp16 it is (research.md R3).
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
print(f"Loaded {BASE_MODEL_ID} in fp16 on {next(model.parameters()).device}")


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. Train

Loss is computed on the completion tokens only (`DataCollatorForCompletionOnlyLM`) — the model is
trained to produce the classification labels, never to reproduce the findings it was given
(FR-087, consistent with research.md R1's decision that this model never generates argumentation
or recommendation prose).


In [ ]:
from trl import DataCollatorForCompletionOnlyLM, SFTTrainer
from transformers import TrainingArguments

collator = DataCollatorForCompletionOnlyLM(RESPONSE_TEMPLATE, tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    seed=SEED,
    data_seed=SEED,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=1,
    optim="adamw_torch",
    report_to=[],
    save_strategy="steps" if USE_VALIDATION_SLICE else "no",
    save_steps=50,
    save_total_limit=1,
    eval_strategy="epoch" if USE_VALIDATION_SLICE else "no",
    load_best_model_at_end=USE_VALIDATION_SLICE,
    metric_for_best_model="eval_loss" if USE_VALIDATION_SLICE else None,
)

callbacks = []
if USE_VALIDATION_SLICE:
    from transformers import EarlyStoppingCallback
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=3))

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    callbacks=callbacks or None,
)


In [ ]:
train_result = trainer.train()
print(train_result)


## 7. Merge the adapter and save locally (git-ignored)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Merged model saved to {OUTPUT_DIR}")
print(list(OUTPUT_DIR.iterdir()))


## 8. Sanity check — NOT the release gate

This runs a couple of training examples back through the merged model so an obviously broken
export (garbage tokens, wrong format) is caught immediately. It is not a substitute for
`eval/replay.py` (SC-015), `eval/heldout.py` (SC-023), or `eval/determinism.py` (SC-016), all of
which run against the deployed Inference Endpoint, not this in-notebook model object.


In [ ]:
merged_model.eval()

for sample in examples[:3]:
    messages = [
        {"role": "system", "content": sample["system"]},
        {"role": "user", "content": sample["prompt"]},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(merged_model.device)

    with torch.no_grad():
        output_ids = merged_model.generate(
            input_ids,
            max_new_tokens=64,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    completion = tokenizer.decode(output_ids[0, input_ids.shape[-1]:], skip_special_tokens=True)

    print(f"record {sample['record_id']}")
    print(f"  expected: {sample['completion']}")
    print(f"  produced: {completion.strip()}")
    print()


## 9. Push to the private Hugging Face repository

Only the finished weights leave this machine (FR-097, FR-105). If `HF_REPO_ID` was left as
`None`, the repository is created under the logged-in account as
`<username>/toxoai-llama32-1b-ft`.


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = HF_REPO_ID
if repo_id is None:
    username = api.whoami()["name"]
    repo_id = f"{username}/toxoai-llama32-1b-ft"

api.create_repo(repo_id, private=HF_REPO_PRIVATE, exist_ok=True)
print(f"Pushing to https://huggingface.co/{repo_id} (private={HF_REPO_PRIVATE})")

merged_model.push_to_hub(repo_id, private=HF_REPO_PRIVATE)
tokenizer.push_to_hub(repo_id, private=HF_REPO_PRIVATE)

print("Done.")


## Next steps

1. Configure a Hugging Face Inference Endpoint on `repo_id` above — see `eval/ENDPOINT.md`
   (dedicated endpoint, scale to zero after 15 minutes idle, greedy decoding).
2. Run the release gates from `eval/` against that endpoint, in this order:
   - `python replay.py --endpoint $ENDPOINT_URL` — SC-015, all 24 historical records, expect
     24/24 (19 of these are training rows; a perfect score here proves no regression, not
     generalisation).
   - `python heldout.py --split split.v1.json --endpoint $ENDPOINT_URL` — SC-023, the 5 records
     this model never saw (17, 5, 13, 21, 24). Report **must** carry `unmeasured_classes`.
   - `python determinism.py --endpoint $ENDPOINT_URL` — SC-016, with and without the response
     cache bypassed; only the bypassed run says anything about the model itself.
3. If the SC-015 replay is not 24/24, do not lower the gate and do not introduce a rules engine
   (FR-084 forbids both). Escalate along the 1B -> 3B -> 8B ladder instead: re-run this notebook
   with `BASE_MODEL_ID` set to a 3B instruct model, switching from LoRA to QLoRA (fp16 LoRA at 3B
   does not fit the 8 GB card).
